In [15]:
import os
from dotenv import load_dotenv
from google.adk.agents import LlmAgent
from google.adk.runners import InMemoryRunner
from quiet import silence
load_dotenv()
silence()

In [16]:
MODEL = "gemini-3.5-flash"

agent = LlmAgent(
    model = MODEL,
    name="assistant",
    instruction="You are a concise, friendly assistant. reply in a single short sentence.",
)

In [17]:
result = await InMemoryRunner(agent=agent).run_debug("Say hello in Spanish", verbose=True)

assistant > ¡Hola! I hope you're having a wonderful day.


In [30]:
import board

board.reset_board()
board.add_goal("Read notes.txt, translate its contents into natural Spanish, and write the Spanish to spanish.txt")
board.list_todos()

[{'id': 1,
  'parent_id': None,
  'task': 'Read notes.txt, translate its contents into natural Spanish, and write the Spanish to spanish.txt',
  'status': 'pending',
  'result': ''}]

In [31]:
board.show_board()

Goal #1: Read notes.txt, translate its contents into natural Spanish, and write the Spanish to spanish.txt

In [32]:
# add tools to the agents
def show_todos() -> list[dict]:
    """List every todo on the board. A goal has parent_id None; a step has parent_id set to its goal's id."""

def plan_steps(goal_id: int, steps: list[str]) -> dict:
    """Break a goal into a ordered checklist of steps on the board. Pass the goal's id and a short list of step descriptions."""
    return {"goal_id": goal_id, "step_ids": [board.add_step(goal_id, step) for step in steps]}

def complete_task(task_id: int, result: str) -> dict:
    """Mark a todo (a step or the goal) with this id as done and record a short result summary."""
    board.complete_todo(task_id, result)
    return {"task_id": task_id, "status": "done"}

In [33]:
board_agent = LlmAgent(
    model = MODEL,
    name="board_agent",
    instruction="You help manage a shared todo board.",
    tools=[show_todos, complete_task]
)

In [34]:
result = await InMemoryRunner(agent=board_agent).run_debug("what is on the board right now, and what is its status?")

board_agent > The todo board is currently empty. There are no goals or steps listed.


In [29]:
import requests
def send_push_notification(message: str) -> str:
    """Send a short push notification to the user's phone to report a result or that a job is done."""
    payload = {
        "token": os.getenv("PUSHOVER_TOKEN"),
        "user": os.getenv("PUSHOVER_USER"),
        "message":message,
    }
    response = requests.post("https://api.pushover.net/1/messages.json", data=payload)
    return f"Push notification sent (status {response.status_code})."

In [36]:
notifier = LlmAgent(
    model=MODEL,
    name="notifier",
    instruction="You notify the user. When asked, use your tool to send a push notification.",
    tools=[send_push_notification],
)

result = await InMemoryRunner(agent=notifier).run_debug("Send a push notification that says hello from google adk.")

notifier > I have sent a push notification saying "hello from google adk".
